## Module submission header
### Submission preparation instructions 
_Completion of this header is mandatory, subject to a 2-point deduction to the assignment._ Only add plain text in the designated areas, i.e., replacing the relevant 'NA's. You must fill out all group member Names and Drexel email addresses in the below markdown list, under header __Module submission group__. It is required to fill out descriptive notes pertaining to any tutoring support received in the completion of this submission under the __Additional submission comments__ section at the bottom of the header. If no tutoring support was received, leave NA in place. You may as well list other optional comments pertaining to the submission at bottom. _Any distruption of this header's formatting will make your group liable to the 2-point deduction._

### Module submission group
- Group member 1
    - Name: Naren Garapati
    - Email: ng658@drexel.edu
- Group member 2
    - Name: NA
    - Email: NA
- Group member 3
    - Name: NA
    - Email: NA
- Group member 4
    - Name: NA
    - Email: NA

### Additional submission comments
- Tutoring support received: NA
- Other (other): NA

# Assignment group 4: Machine learning and regression

## Module B _(62 pts)_ Exploring Classifier Transferability
### Data sets

__Data Set 1:__ There's a lot more than e-mail text out there, and malicious SPAM-like text-based deception is pervasive in other domains. One domain of particular interest to a few companies is called _opinion SPAM_, in which product and business reviews are spoofed, either to help or hurt a business.

An interesting data set for purposes of studying opinion SPAM was produced by a researcher named Myle Ott. In addition to collecting real reviews on hotels from the web and TripAdvisor, Ott et al. ran Amazon Mechanical Turk surveys to have real people write both positive and negative fake reviews of the hotels:

- http://myleott.com/op-spam.html

The goal with the data set was to train computers to detect which reviews were real vs. fake. These are provided in the following nested file structure:

- `./data/op_spam_v1.4/negative_polarity/deceptive_from_MTurk/fold[1-5]/*.txt`
- `./data/op_spam_v1.4/positive_polarity/deceptive_from_MTurk/fold[1-5]/*.txt`
- `./data/op_spam_v1.4/negative_polarity/truthful_from_Web/fold[1-5]/*.txt`
- `./data/op_spam_v1.4/positive_polarity/truthful_from_TripAdvisor/fold[1-5]/*.txt`

__Data Set 2:__ The big picture of what we're trying to do here is train an Opinion SPAM classifier on the _curated_ __Data Set 1__, and apply it to get an idea of how prolific SPAM is on this completely different, _real-world_ hotel [booking website's](booking.com) data. The data from this website live in the assignment's data directory, too:

- `./data/Hotel_Reviews.csv`
    
and were taken from [Kaggle](https://www.kaggle.com/jiashenliu/515k-hotel-reviews-data-in-europe).

__B1.__ _(2 pts_) To load the Op SPAM data we'll be using `sklearn`, but as a requirement we'll need a full list of all the different review files in the data set. To compile a list of file paths, review the datas directory structure and use the `glob` module's `.glob(regex)` method to output a list of all `all_files` matching the provided `regex` pattern.

When this is complete, print the first 5 files to show your code's function.

In [5]:
# code here
import sklearn 

import glob
import numpy as np
import pandas as pd

all_files = glob.glob('./data/op_spam_v1.4/*/*/fold*/*.txt')

all_files[:5]

['./data/op_spam_v1.4/positive_polarity/deceptive_from_MTurk/fold2/d_talbott_9.txt',
 './data/op_spam_v1.4/positive_polarity/deceptive_from_MTurk/fold2/d_talbott_8.txt',
 './data/op_spam_v1.4/positive_polarity/deceptive_from_MTurk/fold2/d_affinia_20.txt',
 './data/op_spam_v1.4/positive_polarity/deceptive_from_MTurk/fold2/d_hardrock_18.txt',
 './data/op_spam_v1.4/positive_polarity/deceptive_from_MTurk/fold2/d_hardrock_19.txt']

__B2.__ _(3 pts)_ Since this is supervised learning, we'll neeed labels, too. To construct, use a regex match on `all_files`. In particular, since we're doing sentiment classification, utilize the word 'positive_polarity' in the file path to indicate a positve label (of value `1`) and otherwise use a negative label (value `0`). Store these values in a `np.array()` called `labels.

When this is done, compute and print the size of positive and negative portions of the data set and discuss the imbalance you observe in the response box below. 

_Response._ 

In [7]:
# code here
import numpy as np
import re

labels = np.array([1 if "positive_polarity" in file else 0 for file in all_files])

num_positive = np.sum(labels)
num_negative = len(labels) - num_positive

percent_positive = (num_positive / len(labels)) * 100
percent_negative = (num_negative / len(labels)) * 100

print(f"Percentage of positive reviews: {percent_positive}%")
print(f"Percentage of negative reviews: {percent_negative}%")


Percentage of positive reviews: 50.0%
Percentage of negative reviews: 50.0%


__B3.__ _(3 pts)_ Now, `import` `sklearn`'s TDM-maker `CountVectorizer` from `sklearn.feature_extraction.text`. Initialize an instance of 
- `CountVectorizer(input = 'filename')` 

and called `vectorizer`, apply its `.fit()` and `.transform()` methods to `all_files` to produce a `TDM`.

When this is complete, exhibit its shape, and be sure to apply `TDM.toarra()` to convert the matrix to a dense representation.

In [8]:
# code here
from sklearn.feature_extraction.text import CountVectorizer


vectorizer = CountVectorizer(input='filename')

# Fit the vectorizer 
vectorizer.fit(all_files)

# Transform the files into a TDM
TDM = vectorizer.transform(all_files)

# Display the shape of the TDM
print(f"TDM shape (documents × terms): {TDM.shape}")

# Convert to dense array as required in the question
TDM_array = TDM.toarray()

# Display information about the dense array
print(f"TDM array shape: {TDM_array.shape}")
print(f"TDM array type: {type(TDM_array)}")

TDM shape (documents × terms): (1600, 9571)
TDM array shape: (1600, 9571)
TDM array type: <class 'numpy.ndarray'>


__B4.__ _(2 pts)_ Now, use `train_test_split` to split the `TDM` and `labels` into $75\%$ training and $25\%$ test sets, importing the function from `sklearn.model_selection`. Also, be sure to use use `random_state = 0`.

In [9]:
# code here
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    TDM_array,  
    labels,     
    test_size=0.20,  
    random_state=42   
)

__B5.__ _(5pts)_ Now, `import`, initialize, and `.fit()` a binary classifer of your choosing (from __Chapter 8.__) with `sklearn` on the training data split. After training, apply and print `.predict()` and `.score()` to review the model's accuracy.

In [10]:
# code here
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='lbfgs', max_iter=1000, multi_class='ovr')

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
results = model.score(X_test, y_test)

print(results)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


0.934375


__B6.__ _(5 pts)_ Now, determine precision, recall, and $F_1$ for the classifier's performance on the test set. Do these results provide any different information as compared to accuracy? If not, why do you think? Provide discussion in the markdown cell below.

_Response._

In [12]:
# code here
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


print(f"Accuracy: {accuracy_score}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

Accuracy: <function accuracy_score at 0x30c43d300>
Precision: 0.9518072289156626
Recall: 0.9239766081871345
F1 Score: 0.9376854599406528

Confusion Matrix:
[[141   8]
 [ 13 158]]


__B7.__ _(2 pts)_ Let's see how well our sentiment polarity classifier does on a different data set:

- `./data/Hotel_Reviews.csv`

which was hosted on a Kaggle competition, but came from from Booking.com:

- https://www.kaggle.com/jiashenliu/515k-hotel-reviews-data-in-europe

There's a decent description of the data there, where it seems a customer can comment with positive and negative reviews, in parallel. To get started, load these data in with pandas, print out the column names and identify (in the markdown cell, below) which have the positive and the negative reviews.

In [17]:
!pip3 install kagglehub


_Response._ 

In [22]:
# code here
import pandas as pd
import kagglehub
path = kagglehub.dataset_download("jiashenliu/515k-hotel-reviews-data-in-europe")


In [23]:
path += '/Hotel_Reviews.csv'

hotel_data = pd.read_csv(path)
hotel_data.head()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,1403,Location was good and staff were ok It is cut...,21,9,7.1,"[' Leisure trip ', ' Family with young childre...",3 days,52.360576,4.915968
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,210,1403,Great location in nice surroundings the bar a...,26,1,3.8,"[' Leisure trip ', ' Solo traveler ', ' Duplex...",3 days,52.360576,4.915968
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/24/2017,7.7,Hotel Arena,New Zealand,You When I booked with your company on line y...,140,1403,Amazing location and building Romantic setting,8,3,6.7,"[' Leisure trip ', ' Couple ', ' Suite ', ' St...",10 days,52.360576,4.915968


__B8.__ _(1 pts)_ Sometimes, a reviewer won't leave a positive or negative review in one of the categories. However, what's left is not a conventional N/A or anything. Refer back to the data dictionary:

- https://www.kaggle.com/jiashenliu/515k-hotel-reviews-data-in-europe

and determine what we should match for to filter out any missing/null reviews.

_Response._

__B9.__ _(7 pts)_ Use your observation from __B8.__ to create a single list with all of the non null reviews, as well as a parallel list of labels: $1$s (for positive review texts) and $0$s (for the negative review texts).

In [27]:
# code here

all_reviews = []
all_labels = []


for positive_review in hotel_data["Positive_Review"]:
    if positive_review != "No Positive":
        all_reviews.append(positive_review)
        all_labels.append(1)

for negative_review in hotel_data["Negative_Review"]:
    if negative_review != "No Negative":
        all_reviews.append(negative_review)
        all_labels.append(0)


import numpy as np
all_labels = np.array(all_labels)

# Print the total number of reviews we collected
print(f"len reviews: {len(all_reviews)}")
print(f"len labels: {len(all_labels)}")


len reviews: 867640
len labels: 867640


__B10.__ _(2 pts)_ How many positive and negatives were there? Does this data set have a class imbalance? Specifically, determine the percentage of reviews that were positive and comment on the presence of any imbalance in the markdown cell below.

_Response._ 

In [30]:
# code here

# just add up the array to get number of positive values
num_positive = np.sum(all_labels)  
# subtract positive from total to get len of negatives
num_negative = len(all_labels) - num_positive 

total_reviews = len(all_labels)

# Calculate percentages
percent_positive = (num_positive / total_reviews) * 100
percent_negative = (num_negative / total_reviews) * 100

#results
print(f"Total number of reviews: {total_reviews}")
print(f"Number of positive reviews: {num_positive} ({percent_positive:.2f}%)")
print(f"Number of negative reviews: {num_negative} ({percent_negative:.2f}%)")



Total number of reviews: 867640
Number of positive reviews: 479792 (55.30%)
Number of negative reviews: 387848 (44.70%)


__B11.__ _(5 pts)_ Use `CountVectorizer()` again&mdash;now to create a TDM for the new hotel data. Note: You must use the same initialized vectorizer from __B3.__, i.e., after is has run `.fit()`. So, here you must start from the `'.transform()'` step. If you re-initialize the vectorizer, you will wind up with a different vocabulary! Note: you also have to change the input format with `vectorizer.input`. It was equal to `'filename'` which would create a TDM by a list of files. Now we want it to work off of a list of strings. This will work if we set:
- `'vectorizer.input = content'`

In [31]:
# code here
vectorizer.input = 'content'

booking_TDM = vectorizer.transform(all_reviews)

In [32]:
booking_TDM.shape

(867640, 9571)

In [33]:
booking_TDM_array = booking_TDM.toarray()
booking_TDM_array.shape

(867640, 9571)

In [34]:
original_vocab_size = TDM.shape[1]
booking_vocab_size = booking_TDM.shape[1]
print(f"Original vocabulary size: {original_vocab_size}")
print(f"Booking.com vocabulary size: {booking_vocab_size}")

Original vocabulary size: 9571
Booking.com vocabulary size: 9571


__B12.__ _(5 pts)_ Apply your Classifier to this new, Booking.com TDM and compute accuracy, precision, recall, and $F_1$. What do you notice? Is there any more of a class imbalance now? Comment in the markdown cell below.

_Response._

In [35]:
# code here

booking_y_pred = model.predict(booking_TDM_array)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

booking_accuracy = accuracy_score(all_labels, booking_y_pred)
booking_precision = precision_score(all_labels, booking_y_pred)
booking_recall = recall_score(all_labels, booking_y_pred)
booking_f1 = f1_score(all_labels, booking_y_pred)

# Print the results
print(f"Performance on Booking.com data:")
print(f"Accuracy: {booking_accuracy}")
print(f"Precision: {booking_precision}")
print(f"Recall: {booking_recall}")
print(f"F1 Score: {booking_f1}")

# Generate a confusion matrix for a more detailed analysis
booking_cm = confusion_matrix(all_labels, booking_y_pred)
print("\nConfusion Matrix:")
print(booking_cm)


# Calculate class distribution in predictions
predicted_positives = np.sum(booking_y_pred)
predicted_negatives = len(booking_y_pred) - predicted_positives
print(f"\nPredicted class distribution:")
print(f"Predicted positive: {predicted_positives} ({predicted_positives/len(booking_y_pred)*100:.2f}%)")
print(f"Predicted negative: {predicted_negatives} ({predicted_negatives/len(booking_y_pred)*100:.2f}%)")

: 

__B13.__ _(2 pts)_  Compare these results with the results from __B6__. Is the performance better or worse in some areas (e.g., precision vs. recall) than others? Do you think our sentiment polarity classifier transferred well from the one Opintion SPAM dataset to this one from Booking.com? Place your discussion in the markdown box below.

_Response._

__B14.__ _(3 pts)_ Go back to the Opinion SPAM data and rebuild the _SPAM_ (no longer sentiment polarity) labels for that dataset's classification, in particular using the patter `deceptive` inside of the file names to produce positive-valued (`1`) labels, and `0`s, otherwise.

In [ ]:
# code here

__B15.__ _(2 pts)_ Now, train your classifier on _all_ of the Opinion SPAM labels. Note: you _must_ initialize a new classifier in order to classify _SPAM_, instead of polarity. However, we can just reuse our `TDM` from __B3__.

In [ ]:
# code here

__B16.__ _(3 pts)_ Run the classifier you just trained on the new hotel reviews data set. Make classification at a threshold of $0.5$ and report the percentage of the new data set that our classifier thinks is SPAM. 

In [ ]:
# code here

__B17.__ (2 pts) Interpret the output percentage from __B16__. Is this a big number? If correct, what would it mean for Booking.com? Do you our classification was a reliable assessment? Why or why not? Place your discussion in the markdown cell, below.

_Response._ 

__C18.__ _(2 pts)_ Sort the Booking.com reviews by their prediction probabilities from high to low. Either use `sorted()` on a list of `(probability, review)` tuples, or create a pandas data frame with the two columns and use the `.sort_values()` method.

In [ ]:
# code here

__B19.__ _(2 pts)_ We really don't have SPAM labels for the Booking.com data. So, inspect the first few most and least spammy reviews. What observations can you draw? Do you see any qualitative differences between the most and least spammy reviews? Do you think the classifier is working?  Place your discussion in the markdown cell, below.

_Response._ 

In [ ]:
# code here

__B20.__ (2 pts) What aspects of our classifier could we modify to potentially improve our SPAM classifier's performance? Specifically, discuss the potential effects to this experiment in selecting or transforming our features, or optimizing any criteria for our predictions in the markdown cell, below.

_Respnose._ 

__B21.__ _(2 pts)_ How could we get an evaluation out of this experiment and _really_ know if the classifer is working? What would we have to do with the Booking.com data in order to get a strong sense of our performance on SPAM? Is there _any_ reasonable labeling of this data that we could come up with, or would we have to get some new data that we have more control over? Place your discussion in the markdown cell, below.

_Response._